# Kernel Overlap Trace

This notebook provides the percentage of overlap for communication and compute kernels. Communication kernels are identified by the 'nccl' prefix.

In [ ]:
import pandas as pd
import plotly.offline as pyo

from IPython.display import display, HTML, Markdown

import nsys_display

pd.options.display.float_format = '{:.1f}'.format

display(HTML("<style>.container { width:95% !important; }</style>"))
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pyo.init_notebook_mode()

## Kernel Matrix

the matrix will display communication-compute kernel matrix, kernels are sorted in duration descending order.   
all communication kernels will be shown, and you can select how many top compute kernels to show.    
first define a formated_display function for better display.


In [ ]:
def formated_display(df):
    col_width = 200

    styles = [
        {
            'selector': 'thead th.col_heading',
            'props': [
                ('position', 'sticky'),
                ('top', '0'),
                ('background-color', 'white'),
                ('z-index', '5'),
                ('white-space', 'normal'),
                ('word-break', 'break-word'),
            ]
        },
        {
            'selector': 'tbody th.row_heading',
            'props': [
                ('position', 'sticky'),
                ('left', '0'),
                ('background-color', 'white'),
                ('z-index', '6'),
            ]
        },
    ]
    df_formated = (
        df.style
           .set_table_styles(styles)
           .format(formatter="{:.2%}", subset=df.columns[1:])
           .set_properties(**{'min-width': f'{col_width}px'})
        )

    display(df_formated)

set the 'top_n_compute_kernels', it means how many top compute kernels you want to display, default is 8.

In [ ]:
top_n_kernels = 8 

df = pd.read_parquet('kernel_matrix_final.parquet')
kernel_size = df.shape[0]
kernel_show_size = min(kernel_size, top_n_kernels)

column_list=list(range(0, 2 + kernel_show_size ))

display the kernel matrix, if the 'total ** overlap' column large than 100%, it's because some duration is overlaped more than 1 time. 

In [ ]:
df_show = df.iloc[0:kernel_show_size , column_list].set_index('shortName')
df_show["total_overlap"] = df_show.iloc[:, 1:].sum(axis=1)
df_show.index.name = None
formated_display(df_show)

The table associates each rank number with the original filename. Ranks are assigned assuming that the file names include the rank with sufficient zero padding for proper sorting. Otherwise, the actual rank may differ from the assigned ID.

In [ ]:
files_df = pd.read_parquet("files.parquet")
display(files_df)